# J 

In [29]:
import os
# os.getcwd()

In [5]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)

conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

cur = conn.cursor()
for table in ["customers", "orders", "order_details", "products", "employees"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}")

customers: 91
orders: 830
order_details: 2155
products: 77
employees: 9


### Exercice 1

In [6]:
query = """
SELECT order_id, ship_name 
FROM orders 
WHERE ship_country='Switzerland' OR ship_country='Argentina' 
    AND freight > 20;
"""
results = cur.execute(query)
for row in results:
    print(row)

(10254, 'Chop-suey Chinese')
(10255, 'Richter Supermarkt')
(10370, 'Chop-suey Chinese')
(10409, 'Océano Atlántico Ltda.')
(10419, 'Richter Supermarkt')
(10448, 'Rancho grande')
(10519, 'Chop-suey Chinese')
(10537, 'Richter Supermarkt')
(10666, 'Richter Supermarkt')
(10716, 'Rancho grande')
(10731, 'Chop-suey Chinese')
(10746, 'Chop-suey Chinese')
(10751, 'Richter Supermarkt')
(10758, 'Richter Supermarkt')
(10828, 'Rancho grande')
(10916, 'Rancho grande')
(10931, 'Richter Supermarkt')
(10937, 'Cactus Comidas para llevar')
(10951, 'Richter Supermarkt')
(10958, 'Océano Atlántico Ltda.')
(10966, 'Chop-suey Chinese')
(10986, 'Océano Atlántico Ltda.')
(11029, 'Chop-suey Chinese')
(11033, 'Richter Supermarkt')
(11041, 'Chop-suey Chinese')
(11075, 'Richter Supermarkt')


### Exercice 2

In [8]:
query = """
SELECT *
FROM orders
WHERE order_date BETWEEN '1997-01-01' AND '1997-12-31'
ORDER BY freight DESC
LIMIT 5;
"""
results = cur.execute(query)
for row in results:
    print(row)

(10540, 'QUICK', 3, '1997-05-19', '1997-06-16', '1997-06-13', 3, 1007.64001, 'QUICK-Stop', 'Taucherstraße 10', 'Cunewalde', None, '01307', 'Germany')
(10691, 'QUICK', 2, '1997-10-03', '1997-11-14', '1997-10-22', 2, 810.049988, 'QUICK-Stop', 'Taucherstraße 10', 'Cunewalde', None, '01307', 'Germany')
(10514, 'ERNSH', 3, '1997-04-22', '1997-05-20', '1997-05-16', 2, 789.950012, 'Ernst Handel', 'Kirchgasse 6', 'Graz', None, '8010', 'Austria')
(10479, 'RATTC', 3, '1997-03-19', '1997-04-16', '1997-03-21', 3, 708.950012, 'Rattlesnake Canyon Grocery', '2817 Milton Dr.', 'Albuquerque', 'NM', '87110', 'USA')
(10612, 'SAVEA', 1, '1997-07-28', '1997-08-25', '1997-08-01', 2, 544.080017, 'Save-a-lot Markets', '187 Suffolk Ln.', 'Boise', 'ID', '83720', 'USA')


### Exercice 3

In [ ]:
query = """
SELECT *
FROM orders
WHERE order_date BETWEEN '1997-01-01' AND '1997-12-31'
ORDER BY freight DESC
LIMIT 5;
"""
results = cur.execute(query)
for row in results:
    print(row)

### Exercice 4

#### Solution 1 : maintenant que sqlite a implémenté `SELECT DISTINCT`

In [17]:
query = """
SELECT DISTINCT ship_country, ship_city
FROM orders;
"""
results = cur.execute(query)
#for row in results:
#    print(row)

print(len(list(results)))

70


In [ ]:
#### Solution 1 : 

In [16]:
query = """
SELECT ship_country, ship_city
FROM orders;
"""

# On récupère les résultats sous forme de list
all_couples = list(cur.execute(query))

# On initialise vide
unique_couples = []

# from collections import Counter
# Counter(all_couples).most_common()

for country_and_city in all_couples:
    if country_and_city not in unique_couples:
        unique_couples.append(country_and_city)


print(len(unique_couples))

70


#### Exercice 4

In [21]:
query = """
SELECT *
FROM orders
WHERE ship_country='Brazil' 
ORDER BY order_date ASC
LIMIT 3;
"""

three_first_orders = list(cur.execute(query))
# print(three_first_orders)


## Pour les observer plus joliment :

import pandas as pd
three_first_orders_df = pd.DataFrame(three_first_orders)
three_first_orders_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,10250,HANAR,4,1996-07-08,1996-08-05,1996-07-12,2,65.830002,Hanari Carnes,"Rua do Paço, 67",Rio de Janeiro,RJ,05454-876,Brazil
1,10253,HANAR,3,1996-07-10,1996-07-24,1996-07-16,2,58.169998,Hanari Carnes,"Rua do Paço, 67",Rio de Janeiro,RJ,05454-876,Brazil
2,10256,WELLI,3,1996-07-15,1996-08-12,1996-07-17,2,13.970000,Wellington Importadora,"Rua do Mercado, 12",Resende,SP,08737-363,Brazil


#### Exercice 5

In [24]:
query = """
SELECT *
FROM orders
WHERE ship_region IS NULL 
ORDER BY shipped_date DESC;
"""

print(len(list(cur.execute(query))))

507


#### Exercice 6

In [26]:
query = """
SELECT * FROM orders 
WHERE ship_name LIKE 'S%' ; 
"""
# Ou : WHERE ship_name LIKE 'S%' OR ship_name LIKE 's%';

print(list(cur.execute(query))[:2])


[(10252, 'SUPRD', 4, '1996-07-09', '1996-08-06', '1996-07-11', 2, 51.2999992, 'Suprêmes délices', 'Boulevard Tirou, 255', 'Charleroi', None, 'B-6000', 'Belgium'), (10271, 'SPLIR', 6, '1996-08-01', '1996-08-29', '1996-08-30', 2, 4.53999996, 'Split Rail Beer & Ale', 'P.O. Box 555', 'Lander', 'WY', '82520', 'USA')]


### Exercice 7

In [27]:
query = """
SELECT * 
FROM orders 
WHERE ship_country = 'UK' OR ship_country = 'USA'
AND freight > 50 
AND order_date < '1998-01-01' ;
"""
print(list(cur.execute(query))[:2])

[(10272, 'RATTC', 6, '1996-08-02', '1996-08-30', '1996-08-06', 2, 98.0299988, 'Rattlesnake Canyon Grocery', '2817 Milton Dr.', 'Albuquerque', 'NM', '87110', 'USA'), (10289, 'BSBEV', 7, '1996-08-26', '1996-09-23', '1996-08-28', 3, 22.7700005, "B's Beverages", 'Fauntleroy Circus', 'London', None, 'EC2 5NT', 'UK')]


#### Exercice 8

In [28]:
query = """
SELECT order_id, order_date AS DateOfOrder, ship_name
FROM orders
WHERE order_date > '1997-01-01' 
    AND ship_region IS NULL 
    AND ship_country NOT IN ('USA', 'UK')
ORDER BY order_date ASC
LIMIT 10;
"""

print(list(cur.execute(query))[:2])

[(10402, '1997-01-02', 'Ernst Handel'), (10403, '1997-01-03', 'Ernst Handel')]
